# Shared weather vs shared water — upstream precipitation at seeds 13, 17

The distance-preserving control showed the gain comes from **nearby** basins, not the true upstream
topology. That leaves "proximity" as an uncharacterised residual. This notebook runs the cheapest
positive decomposition of it.

A neighbour's discharge could help because (1) nearby basins share **weather** — and the baseline
already gets its own precipitation, so upstream precipitation is a spatial smooth of a channel it
has — or (2) discharge carries **catchment state** (soil moisture, baseflow, snowmelt storage) that
precipitation does not. The contrast `realizable − upPrecip` isolates (2).

Seed 11 is already on disk: `L_upPrecip` +0.016 vs realizable +0.034 on connected basins, so
precipitation recovers ~47% of the gain. **This notebook adds seeds 13 and 17** so the decomposition
is 3-seed like every other load-bearing claim.

Pre-registration: `experiments/topology_ablation/preregistration_precip_decomposition.md`.

**Runtime → Change runtime type → T4 GPU → Run all.** ~80 min.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEEDS = [13, 17])

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[13,17]  # seed 11 already on disk
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydro_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('CAMELS:', DRIVE_CAMELS_PATH); print('RUNS  :', DRIVE_RUNS); print('SEEDS :', SEEDS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(f'{DRIVE_RUNS}/topology_ablation/component0', exist_ok=True)
print('datasets ->', os.path.realpath(RD)); print('runs     ->', os.path.realpath(RR))

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build the upstream-precipitation feature

Area-weighted mean of upstream **precipitation**, lag 1. Seed-independent (observed forcings), so one
build serves both seeds. The column is named `upstream_q` so the config is byte-identical to the other
structural conditions apart from the feature file. Guarded on a `'date'`-named index.

In [ ]:
%cd {REPO_DIR}
import pickle
FEAT='experiments/topology_ablation/features'
def named_ok(p):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb')); return d[next(iter(d))].index.name=='date'
fp=f'{FEAT}/upstream_precip_component0_lag1.p'
if not named_ok(fp):
    !python experiments/topology_ablation/build_upstream_variants.py --network component0 --variant precip --lag-days 1
else:
    print('upstream_precip feature already present with date-named index')
d=pickle.load(open(fp,'rb'))
k=next(iter(d))
print('basins:', len(d), '| index name:', d[k].index.name, '| cols:', list(d[k].columns))

## Cell 8 — Train `L_upPrecip` at seeds 13 and 17

Idempotent: skips a seed whose `test/model_epoch030/test_metrics.csv` already exists. The runner
trains, evaluates, and renames the timestamped directory to the canonical name.

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
for s in SEEDS:
    if done('L_upPrecip',s):
        print(f'seed {s}: already done, skipping'); continue
    print(f'=== training L_upPrecip seed {s} ===')
    !python experiments/topology_ablation/run_upstream_feature.py \
        --network component0 --seed {s} --device cuda:0 --epochs 30 \
        --feature-file experiments/topology_ablation/features/upstream_precip_component0_lag1.p \
        --cond-name L_upPrecip
for s in SEEDS: print(f'  L_upPrecip seed {s} complete:', done('L_upPrecip',s))

## Cell 9 — Verdict: how much of the gain is shared weather?

Pooled 3-seed paired medians on the forward-connected basins (n=150/seed), plus the paired
`realizable − upPrecip` contrast, which is the quantity the pre-registration is about.

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle
from scipy.stats import wilcoxon
B=f'{REPO_DIR}/runs/topology_ablation/component0'; ALL=[11]+SEEDS
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
feat=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
conn=sorted([b for b,v in feat.items() if float(np.nanmax(np.abs(v.values)))>0])
print(f'connected basins: {len(conn)}\n')
rows=[]; pooled={'P':[], 'R':[], 'diff':[]}
for s in ALL:
    L=nse('L',s); P=nse('L_upPrecip',s); R=nse('L_upQpred',s)
    if L is None or P is None or R is None:
        print(f'seed {s}: missing run, skipped'); continue
    idx=[b for b in conn if b in L.index and b in P.index and b in R.index]
    dP=(P[idx]-L[idx]).values; dR=(R[idx]-L[idx]).values
    pooled['P']+=list(dP); pooled['R']+=list(dR); pooled['diff']+=list(dR-dP)
    rows.append((s,len(idx),np.median(dP),np.median(dR),np.median(dR-dP)))
print('| seed |   n | upPrecip Δ | realizable Δ | realizable−upPrecip |')
print('|------|-----|------------|--------------|---------------------|')
for s,n,p_,r_,d_ in rows: print(f'| {s:4d} | {n:3d} |   {p_:+.4f}  |    {r_:+.4f}   |       {d_:+.4f}       |')
P=np.median(pooled['P']); R=np.median(pooled['R']); D=np.median(pooled['diff'])
pw=wilcoxon(pooled['diff'],alternative='greater')[1] if len(pooled['diff'])>10 else float('nan')
share=P/R if R!=0 else float('nan')
print(f'\nPOOLED (n={len(pooled["P"])} basin-seed):')
print(f'  upPrecip Δ            = {P:+.4f}')
print(f'  realizable Δ          = {R:+.4f}')
print(f'  realizable − upPrecip = {D:+.4f}   (paired Wilcoxon one-sided p={pw:.2e})')
print(f'  shared-weather share  = P/R = {share*100:.0f}%')
allpos = all(r[4]>0 for r in rows)
print('\n=== PRE-REGISTERED VERDICT ===')
if pw<0.05 and allpos and share<=0.85:
    print(f'  DISCHARGE ADDS BEYOND WEATHER. ~{share*100:.0f}% of the gain is upstream precipitation;')
    print(f'  the remaining ~{(1-share)*100:.0f}% needs discharge. Report both rows in the paper.')
elif share>0.85 or pw>=0.05:
    print('  FALSIFIED: the gain is (mostly) weather smoothing. Upstream precipitation matches')
    print('  the realizable model, so the two-stage discharge machinery is not earning its place.')
    print('  This must be stated in the abstract and Discussion.')
else:
    print('  MIXED — inspect per-seed rows above before writing.')

## Cell 10 — Persistence check (did the runs land in Drive?)

In [ ]:
print('=== persistence (in Drive?) ===')
for s in SEEDS:
    dp=f'{DRIVE_RUNS}/topology_ablation/component0/L_upPrecip_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    print(f'  L_upPrecip seed {s}: {os.path.isfile(dp)}')
print('\nIf any say False the run is only in the VM and will be lost on recycle.')

## Done

Two runs (`L_upPrecip` seeds 13, 17) persist to Drive. Report the **Cell 9 verdict** and the
**Cell 10 persistence** lines back.

This closes the paper's largest open gap: it turns the mechanism argument from elimination
("not topology, not direction, not depth") into a decomposition (shared weather vs shared water),
and it is the experiment the current Conclusion describes as future work.